# Лабораторная работа № 3


## Подготовка

In [5]:
import os, json, shutil, logging, random, sys, importlib
from pathlib import Path
from typing import Iterator
from logging.handlers import RotatingFileHandler

random.seed(42)

## Задание 1. Функция и процедура

In [6]:
def log_error(path: str, reason: str) -> None:
    """Логирует путь к файлу и причину ошибки; полезное значение не возвращает."""
    logging.warning("Ошибка в файле %s: %s", path, reason)

def parse_int_safe(value: str) -> int | None:
    """Преобразует строку в целое число или возвращает None при ValueError."""
    try:
        return int(value)
    except ValueError:
        return None

logging.basicConfig(level=logging.WARNING, format="%(levelname)s: %(message)s")
log_error("data.txt", "не удалось прочитать")
print(parse_int_safe("123"), parse_int_safe("abc"))
# Процедура выполняет действие; функция вычисляет и возвращает результат.

123 None


## Задание 2. Типы исключений и MRO

In [7]:
tests = [
    ("1/0", lambda: 1/0),
    ('int("abc")', lambda: int("abc")),
    ("[1, 2][5]", lambda: [1, 2][5]),
    ('{"a": 1}["b"]', lambda: {"a": 1}["b"]),
    ('"5" + 5', lambda: "5" + 5),
    ('open("no_such_file.txt")', lambda: open("no_such_file.txt", encoding="utf-8")),
]
for expression, operation in tests:
    try:
        value = operation()
        if hasattr(value, "close"): value.close()
        print(expression, "->", value)
    except (ArithmeticError, LookupError, OSError, ValueError, TypeError) as exc:
        print(expression, "->", type(exc).__name__)
        print("MRO:", " -> ".join(c.__name__ for c in type(exc).__mro__))
# Например: ZeroDivisionError наследуется от ArithmeticError; KeyError и IndexError — LookupError;
# FileNotFoundError — OSError; ValueError и TypeError — непосредственно от Exception.

1/0 -> ZeroDivisionError
MRO: ZeroDivisionError -> ArithmeticError -> Exception -> BaseException -> object
int("abc") -> ValueError
MRO: ValueError -> Exception -> BaseException -> object
[1, 2][5] -> IndexError
MRO: IndexError -> LookupError -> Exception -> BaseException -> object
{"a": 1}["b"] -> KeyError
MRO: KeyError -> LookupError -> Exception -> BaseException -> object
"5" + 5 -> TypeError
MRO: TypeError -> Exception -> BaseException -> object
open("no_such_file.txt") -> FileNotFoundError
MRO: FileNotFoundError -> OSError -> Exception -> BaseException -> object


## Задание 3. try / except / else / finally

In [8]:
def safe_divide(a, b) -> float | None:
    """Делит a на b; при делении на ноль записывает ошибку и возвращает None."""
    try:
        result = a / b
    except ZeroDivisionError as exc:
        logging.error("Деление на ноль: %s", exc)
        return None
    else:
        print("Деление выполнено успешно")
        return result
    finally:
        print("Операция завершена")

for pair in [(10, 2), (10, 0), (10, "x")]:
    try: print(pair, "->", safe_divide(*pair))
    except TypeError as exc: print(pair, "-> TypeError:", exc)
# finally выполняется и перед выходом через return.

ERROR: Деление на ноль: division by zero


Деление выполнено успешно
Операция завершена
(10, 2) -> 5.0
Операция завершена
(10, 0) -> None
Операция завершена
(10, 'x') -> TypeError: unsupported operand type(s) for /: 'int' and 'str'


## Задание 4. Иерархия собственных исключений

In [9]:
class IngestError(Exception):
    """Базовая ошибка импорта."""

class ConfigError(IngestError):
    """Неверная конфигурация."""

class DocumentError(IngestError):
    """Ошибка одного документа."""
    def __init__(self, path, reason):
        self.path, self.reason = str(path), reason
        super().__init__(f"{self.path}: {self.reason}")

for err in [ConfigError("неверные настройки"), DocumentError("a.json", "нет поля id")]:
    try: raise err
    except IngestError as exc: print(type(exc).__name__, exc)
# except IngestError перехватывает оба исключения-наследника.

ConfigError неверные настройки
DocumentError a.json: нет поля id


## Задание 5. LBYL и EAFP

In [10]:
def read_lbyl(path: str | Path) -> str | None:
    """Сначала проверяет наличие файла, затем читает его."""
    path = Path(path)
    if path.exists() and path.is_file():
        return path.read_text(encoding="utf-8")
    return None

def read_eafp(path: str | Path) -> str | None:
    """Пытается открыть файл и обрабатывает отсутствие."""
    try: return Path(path).read_text(encoding="utf-8")
    except FileNotFoundError: return None

demo = Path("lbyl_eafp_demo.txt")
demo.write_text("Пример", encoding="utf-8")
print(read_lbyl(demo), read_eafp(demo), read_eafp("missing.txt"))
# EAFP надёжнее в гонке состояний: файл может исчезнуть после exists().

Пример Пример None


## Задание 6. Режимы файлов

In [11]:
p = Path("test.txt")
with p.open("w", encoding="utf-8") as f: f.write("Первая\nВторая\nТретья\n")
print(p.read_text(encoding="utf-8"))
with p.open("r", encoding="utf-8") as f:
    for line in f: print(line.rstrip())
with p.open("a", encoding="utf-8") as f: f.write("Четвёртая\n")
print("Число строк:", len(p.read_text(encoding="utf-8").splitlines()))
# Повторное открытие в режиме w усечёт файл и удалит старое содержимое.

Первая
Вторая
Третья

Первая
Вторая
Третья
Число строк: 4


## Задание 7. JSON

In [12]:
data = {"название": "Обработка документов", "список": [1, 2, 3], "текст": "Привет"}
p = Path("sample.json")
with p.open("w", encoding="utf-8") as f: json.dump(data, f, ensure_ascii=False, indent=2)
with p.open("r", encoding="utf-8") as f: loaded = json.load(f)
print("Совпадают:", data == loaded, loaded)
try: json.loads('{"broken": }')
except json.JSONDecodeError as exc: print("Пойман JSONDecodeError:", exc)

Совпадают: True {'название': 'Обработка документов', 'список': [1, 2, 3], 'текст': 'Привет'}
Пойман JSONDecodeError: Expecting value: line 1 column 12 (char 11)


## Задание 8. pathlib

In [13]:
base = Path("pathlib_demo")
raw, processed = base / "data" / "raw", base / "data" / "processed"
raw.mkdir(parents=True, exist_ok=True)
processed.mkdir(parents=True, exist_ok=True)
(raw / "a.json").write_text('{"id": 1}', encoding="utf-8")
(raw / "b.json").write_text('{"id": 2}', encoding="utf-8")
for file in sorted(base.rglob("*.json")):
    print(file.name, file.stat().st_size, file.resolve())

a.json 9 D:\maestria\I semestre\python-labs-gutierrez\pathlib_demo\data\raw\a.json
b.json 9 D:\maestria\I semestre\python-labs-gutierrez\pathlib_demo\data\raw\b.json


## Задание 9. Модуль jsonutils.py

In [14]:
module_text = '''
import json
from pathlib import Path

class DocumentError(Exception):
    """Ошибка чтения JSON."""

def read_json(path: str | Path) -> dict:
    """Читает JSON-словарь или поднимает DocumentError."""
    try:
        with Path(path).open(encoding="utf-8") as f:
            data = json.load(f)
    except (OSError, json.JSONDecodeError) as exc:
        raise DocumentError(f"{path}: {exc}") from exc
    if not isinstance(data, dict):
        raise DocumentError(f"{path}: JSON должен быть объектом")
    return data

def write_json(data: dict, path: str | Path) -> None:
    """Записывает словарь в JSON, сохраняя кириллицу."""
    with Path(path).open("w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

if __name__ == "__main__":
    write_json({"id": 1, "name": "Пример"}, "jsonutils_demo.json")
    print(read_json("jsonutils_demo.json"))
'''
Path("jsonutils.py").write_text(module_text.strip() + "\n", encoding="utf-8")
importlib.invalidate_caches()
import jsonutils
jsonutils.write_json({"id": 1, "name": "Тест"}, "jsonutils_demo.json")
print(jsonutils.read_json("jsonutils_demo.json"))

{'id': 1, 'name': 'Тест'}


## Задание 10. Логирование с уровнями и ротацией

In [15]:
def setup_task10_logging():
    """Консольный INFO и файловый DEBUG с ротацией app.log."""
    logger = logging.getLogger("task10")
    logger.setLevel(logging.DEBUG); logger.handlers.clear(); logger.propagate = False
    console = logging.StreamHandler(); console.setLevel(logging.INFO)
    file_handler = RotatingFileHandler("app.log", maxBytes=100_000, backupCount=3, encoding="utf-8")
    file_handler.setLevel(logging.DEBUG)
    fmt = logging.Formatter("%(asctime)s %(levelname)s %(name)s %(message)s")
    console.setFormatter(fmt); file_handler.setFormatter(fmt)
    logger.addHandler(console); logger.addHandler(file_handler)
    return logger

log10 = setup_task10_logging()
for level, message in [(10,"DEBUG"),(20,"INFO"),(30,"WARNING"),(40,"ERROR"),(50,"CRITICAL")]:
    log10.log(level, message)
print("DEBUG отсутствует в консоли, потому что консольный уровень — INFO.")
print(Path("app.log").read_text(encoding="utf-8"))

2026-10-09 13:11:38,418 INFO task10 INFO
2026-10-09 13:11:38,419 WARNING task10 WARNING
2026-10-09 13:11:38,420 ERROR task10 ERROR
2026-10-09 13:11:38,420 CRITICAL task10 CRITICAL


DEBUG отсутствует в консоли, потому что консольный уровень — INFO.
2026-10-09 13:11:38,416 DEBUG task10 DEBUG
2026-10-09 13:11:38,418 INFO task10 INFO
2026-10-09 13:11:38,419 WARNING task10 WARNING
2026-10-09 13:11:38,420 ERROR task10 ERROR
2026-10-09 13:11:38,420 CRITICAL task10 CRITICAL



## Общая инициализация данных для заданий 11–20

In [16]:
shutil.rmtree("inbox", ignore_errors=True)
Path("inbox/good").mkdir(parents=True)
Path("inbox/nested").mkdir(parents=True)
Path("inbox/good/a.json").write_text('{"id": 1, "name": "Alice", "age": 30}', encoding="utf-8")
Path("inbox/good/b.json").write_text('{"id": 2, "name": "Bob", "age": 25}', encoding="utf-8")
Path("inbox/nested/c.json").write_text('{"id": 3, "name": "Carol", "age": 28}', encoding="utf-8")
Path("inbox/broken.json").write_text('{"id": 4, "name": }', encoding="utf-8")
Path("inbox/no_keys.json").write_text('{"foo": "bar"}', encoding="utf-8")
Path("inbox/not_object.json").write_text('[1, 2, 3]', encoding="utf-8")
print("JSON-файлов:", len(list(Path("inbox").rglob("*.json"))))

JSON-файлов: 6


## Задание 11. Настройки из окружения

In [17]:
class Settings:
    """Хранит источник, путь отчёта и обязательные поля."""
    def __init__(self, source_dir=Path("inbox"), output_path=Path("catalog.json"),
                 required_keys=("id", "name")):
        self.source_dir, self.output_path = Path(source_dir), Path(output_path)
        self.required_keys = tuple(required_keys)

    @classmethod
    def from_env(cls):
        """Читает переменные окружения и проверяет настройки."""
        src = Path(os.environ.get("SOURCE_DIR", "inbox"))
        out = Path(os.environ.get("OUTPUT_PATH", "catalog.json"))
        keys = tuple(k.strip() for k in os.environ.get("REQUIRED_KEYS", "id,name").split(",") if k.strip())
        if not src.is_dir(): raise ConfigError(f"Каталог-источник не существует: {src}")
        if not keys: raise ConfigError("REQUIRED_KEYS не должен быть пустым")
        return cls(src, out, keys)

settings = Settings.from_env()
print(settings.source_dir, settings.output_path, settings.required_keys)

inbox catalog.json ('id', 'name')


## Задание 12. Рекурсивный обход

In [18]:
class Ingester:
    """Пакетно импортирует JSON-документы с изоляцией ошибок файлов."""
    def __init__(self, settings: Settings, logger=None):
        self.settings = settings
        self.logger = logger or logging.getLogger("ingester")

    def iter_files(self) -> Iterator[Path]:
        """Возвращает отсортированный итератор JSON-файлов из подкаталогов."""
        return iter(sorted(self.settings.source_dir.rglob("*.json")))

print([str(p) for p in Ingester(settings).iter_files()])
# rglob компактно выполняет рекурсивный поиск по шаблону; os.walk полезен для более тонкого контроля обхода.

['inbox\\broken.json', 'inbox\\good\\a.json', 'inbox\\good\\b.json', 'inbox\\nested\\c.json', 'inbox\\no_keys.json', 'inbox\\not_object.json']


## Задание 13. Чтение и валидация документа

In [19]:
def load_document(self, path: Path) -> dict:
    """Читает JSON и проверяет словарь и обязательные поля."""
    try:
        with path.open("r", encoding="utf-8") as f: data = json.load(f)
    except (OSError, json.JSONDecodeError) as exc:
        raise DocumentError(path, str(exc)) from exc
    if not isinstance(data, dict):
        raise DocumentError(path, "JSON должен быть объектом (словарём)")
    missing = [key for key in self.settings.required_keys if key not in data]
    if missing: raise DocumentError(path, "отсутствуют поля: " + ", ".join(missing))
    return data

Ingester.load_document = load_document
for path in Ingester(settings).iter_files():
    try: print(path, "->", Ingester(settings).load_document(path))
    except DocumentError as exc: print("Пропущен:", exc)

Пропущен: inbox\broken.json: Expecting value: line 1 column 19 (char 18)
inbox\good\a.json -> {'id': 1, 'name': 'Alice', 'age': 30}
inbox\good\b.json -> {'id': 2, 'name': 'Bob', 'age': 25}
inbox\nested\c.json -> {'id': 3, 'name': 'Carol', 'age': 28}
Пропущен: inbox\no_keys.json: отсутствуют поля: id, name
Пропущен: inbox\not_object.json: JSON должен быть объектом (словарём)


## Задание 14. Конвейер с изоляцией ошибок

In [20]:
def run(self) -> dict:
    """Импортирует все файлы, продолжая работу после ошибки отдельного документа."""
    files = list(self.iter_files()); records, errors = [], []
    for path in files:
        try: records.append(self.load_document(path))
        except DocumentError as exc:
            self.logger.warning("Документ пропущен: %s", exc)
            errors.append({"path": str(path), "reason": exc.reason})
        except Exception as exc:
            self.logger.exception("Неожиданная ошибка в %s", path)
            errors.append({"path": str(path), "reason": str(exc)})
    return {"source": str(self.settings.source_dir), "total_files": len(files),
            "imported": len(records), "skipped": len(errors), "errors": errors, "records": records}

Ingester.run = run
print(json.dumps(Ingester(settings).run(), ensure_ascii=False, indent=2))

{
  "source": "inbox",
  "total_files": 6,
  "imported": 3,
  "skipped": 3,
  "errors": [
    {
      "path": "inbox\\broken.json",
      "reason": "Expecting value: line 1 column 19 (char 18)"
    },
    {
      "path": "inbox\\no_keys.json",
      "reason": "отсутствуют поля: id, name"
    },
    {
      "path": "inbox\\not_object.json",
      "reason": "JSON должен быть объектом (словарём)"
    }
  ],
  "records": [
    {
      "id": 1,
      "name": "Alice",
      "age": 30
    },
    {
      "id": 2,
      "name": "Bob",
      "age": 25
    },
    {
      "id": 3,
      "name": "Carol",
      "age": 28
    }
  ]
}


## Задание 15. Сохранение отчёта

In [21]:
def _save(self, report: dict) -> None:
    """Сохраняет JSON-отчёт; ошибку записи оборачивает в ConfigError."""
    try:
        self.settings.output_path.parent.mkdir(parents=True, exist_ok=True)
        with self.settings.output_path.open("w", encoding="utf-8") as f:
            json.dump(report, f, ensure_ascii=False, indent=2)
        self.logger.info("Отчёт сохранён: %s", self.settings.output_path)
    except OSError as exc:
        raise ConfigError(f"Не удалось сохранить отчёт: {exc}") from exc

Ingester._save = _save

## Задание 16. Логирование конвейера

In [22]:
def setup_logging():
    """Настраивает INFO для консоли и DEBUG для ingest.log, с ротацией 100 КБ и 3 архивами."""
    logger = logging.getLogger("ingester")
    logger.setLevel(logging.DEBUG); logger.handlers.clear(); logger.propagate = False
    console = logging.StreamHandler(); console.setLevel(logging.INFO)
    fh = RotatingFileHandler("ingest.log", maxBytes=100*1024, backupCount=3, encoding="utf-8")
    fh.setLevel(logging.DEBUG)
    fmt = logging.Formatter("%(asctime)s %(levelname)s %(name)s %(message)s")
    console.setFormatter(fmt); fh.setFormatter(fmt)
    logger.addHandler(console); logger.addHandler(fh)
    return logger

pipeline_logger = setup_logging()

## Задание 17. CLI и коды завершения

In [25]:
def main() -> int:
    """Запускает импорт; возвращает 0 при успехе, 1 при ошибке импорта, 2 при ошибке настроек."""
    logger = setup_logging()
    try: cfg = Settings.from_env()
    except ConfigError as exc:
        logger.error("Конфигурация невалидна: %s", exc); return 2
    try:
        ingester = Ingester(cfg, logger)
        report = ingester.run()
        ingester._save(report)
    except IngestError as exc:
        logger.error("Импорт прерван: %s", exc); return 1
    logger.info("Готово: импортировано %d из %d", report["imported"], report["total_files"])
    return 0

# Планировщик может различать успешный запуск и разные типы сбоев по коду завершения.

## Задание 18. Проверка сценариев

In [26]:
exit_code = main()
report = json.loads(Path("catalog.json").read_text(encoding="utf-8"))
print("Код:", exit_code, "| найдено:", report["total_files"],
      "| импортировано:", report["imported"], "| пропущено:", report["skipped"])
print("Импортированные записи:", [r["name"] for r in report["records"]])
print("catalog.json:", Path("catalog.json").exists(), "| ingest.log:", Path("ingest.log").exists())
assert (report["total_files"], report["imported"], report["skipped"], exit_code) == (6, 3, 3, 0)

2026-10-09 13:12:56,260 WARNING ingester Документ пропущен: inbox\broken.json: Expecting value: line 1 column 19 (char 18)
2026-10-09 13:12:56,262 WARNING ingester Документ пропущен: inbox\no_keys.json: отсутствуют поля: id, name
2026-10-09 13:12:56,263 WARNING ingester Документ пропущен: inbox\not_object.json: JSON должен быть объектом (словарём)
2026-10-09 13:12:56,267 INFO ingester Отчёт сохранён: catalog.json
2026-10-09 13:12:56,269 INFO ingester Готово: импортировано 3 из 6


Код: 0 | найдено: 6 | импортировано: 3 | пропущено: 3
Импортированные записи: ['Alice', 'Bob', 'Carol']
catalog.json: True | ingest.log: True


## Задание 19. `.env` и `.gitignore`

In [27]:
Path(".env").write_text("SOURCE_DIR=inbox\nOUTPUT_PATH=catalog.json\nREQUIRED_KEYS=id,name\n", encoding="utf-8")
Path(".gitignore").write_text(
    ".env\nvenv/\n__pycache__/\n*.pyc\n*.log\n.idea/\n.vscode/\ninbox/\n.venv/\n.ipynb_checkpoints/\n",
    encoding="utf-8"
)
try:
    from dotenv import load_dotenv
    load_dotenv(".env", override=True)
    print("Настройки загружены через python-dotenv.")
except ImportError:
    for line in Path(".env").read_text(encoding="utf-8").splitlines():
        if "=" in line:
            key, value = line.split("=", 1); os.environ[key] = value
    print("python-dotenv отсутствует; переменные загружены резервным способом.")
print("Созданы .env и .gitignore. Не добавляйте .env в репозиторий.")

python-dotenv отсутствует; переменные загружены резервным способом.
Созданы .env и .gitignore. Не добавляйте .env в репозиторий.


## Задание 20. Единый pipeline

In [28]:
def generate_report(input_dir: str, output_path: str, required_keys: tuple) -> dict:
    """Создаёт настройки, запускает Ingester, сохраняет и возвращает отчёт."""
    cfg = Settings(input_dir, output_path, required_keys)
    if not cfg.source_dir.is_dir(): raise ConfigError(f"Нет каталога: {cfg.source_dir}")
    if not cfg.required_keys: raise ConfigError("Пустой список обязательных полей")
    logger = setup_logging()
    ingester = Ingester(cfg, logger)
    report = ingester.run()
    ingester._save(report)
    return report

def main_pipeline() -> int:
    """Оркестратор итогового конвейера: настройки, запуск и сводка."""
    logger = setup_logging()
    try:
        cfg = Settings.from_env()
        report = generate_report(str(cfg.source_dir), str(cfg.output_path), cfg.required_keys)
    except ConfigError as exc:
        logger.error("Ошибка конфигурации/сохранения: %s", exc); return 2
    except IngestError as exc:
        logger.error("Импорт прерван: %s", exc); return 1
    logger.info("Итог: %d импортировано, %d пропущено из %d",
                report["imported"], report["skipped"], report["total_files"])
    print("Отчёт:", cfg.output_path.resolve())
    print("Лог:", Path("ingest.log").resolve())
    return 0

final_code = main_pipeline()
print("Код завершения:", final_code)
assert final_code == 0

2026-10-09 13:13:14,846 WARNING ingester Документ пропущен: inbox\broken.json: Expecting value: line 1 column 19 (char 18)
2026-10-09 13:13:14,849 WARNING ingester Документ пропущен: inbox\no_keys.json: отсутствуют поля: id, name
2026-10-09 13:13:14,850 WARNING ingester Документ пропущен: inbox\not_object.json: JSON должен быть объектом (словарём)
2026-10-09 13:13:14,854 INFO ingester Отчёт сохранён: catalog.json
2026-10-09 13:13:14,855 INFO ingester Итог: 3 импортировано, 3 пропущено из 6


Отчёт: D:\maestria\I semestre\python-labs-gutierrez\catalog.json
Лог: D:\maestria\I semestre\python-labs-gutierrez\ingest.log
Код завершения: 0
